# 1/3 - EDA Dataset Awal dan Dataset Canonical

Notebook ini menjadi sumber data canonical. Output wajib notebook ini adalah `dataset_final_manifest.csv`.

Urutan fungsi:
1. membaca dataset awal per kelas;
2. audit distribusi kelas;
3. deduplikasi tahap pertama pada gambar original;
4. preprocessing citra ke grayscale, padding, dan resize 128 Ãƒâ€” 128;
5. deduplikasi tahap kedua setelah preprocessing;
6. menyimpan `dataset_final_manifest.csv` sebagai dasar notebook 2 dan 3.

Catatan metodologis: seluruh notebook setelah ini harus membaca manifest final yang sama agar jumlah data, path gambar, dan label kelas konsisten.

# EDA Dataset Awal - Versi Konsisten Final

Notebook ini menjadi sumber canonical untuk seluruh pipeline EDA.

Output wajib:
1. `dataset_cleaned_preprocessed` sebagai folder dataset final setelah preprocessing dan deduplikasi tahap kedua.
2. `dataset_final_manifest.csv` sebagai manifest resmi yang wajib dibaca oleh notebook fitur dan post-processing.

Dengan desain ini, jumlah data pada notebook 2 dan 3 harus selalu sama dengan jumlah baris pada `dataset_final_manifest.csv`.


In [ ]:
import os
import shutil
from pathlib import Path

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from tqdm.auto import tqdm
from joblib import Parallel, delayed
import imagehash

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120


In [ ]:
# =========================
# KONFIGURASI UTAMA
# =========================
# Semua notebook berikutnya harus memakai folder output yang sama.
def find_project_root(marker="dataset 10 kelas"):
    """Cari root repo dari current working directory notebook."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Folder {marker!r} tidak ditemukan dari {start} atau parent-nya.")

PROJECT_ROOT = find_project_root()
DATASET_ROOT = PROJECT_ROOT / "dataset 10 kelas"                 # folder dataset mentah: class/image
OUTPUT_DIR = Path("dataset_cleaned_preprocessed")                 # folder canonical setelah preprocessing
FINAL_MANIFEST_CSV = Path("dataset_final_manifest.csv")           # manifest final setelah deduplikasi tahap 2

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
TARGET_SIZE = (128, 128)

# Jika True, folder OUTPUT_DIR akan dihapus ulang sebelum preprocessing agar tidak ada file sisa lama.
# Aman jika OUTPUT_DIR memang folder hasil generate, bukan dataset mentah.
RESET_OUTPUT_DIR = True

# pHash exact duplicate: 0. Untuk near-duplicate ringan bisa 2-4, tetapi laporan harus menyebut threshold-nya.
PHASH_DISTANCE_THRESHOLD = 0
HASH_WORKERS = max(1, (os.cpu_count() or 2) - 1)
PREPROCESS_WORKERS = max(1, (os.cpu_count() or 2) - 1)

print("Project root :", PROJECT_ROOT)
print("Dataset root :", DATASET_ROOT)
print("Output dir   :", OUTPUT_DIR)
print("Target size  :", TARGET_SIZE)
print("Hash workers :", HASH_WORKERS)


In [ ]:
def create_initial_audit(root_path, extensions=IMAGE_EXTENSIONS):
    """Membaca struktur folder class/image secara deterministik."""
    root_path = Path(root_path)
    if not root_path.exists():
        raise FileNotFoundError(
            f"Folder dataset tidak ditemukan: {root_path.resolve()}\n"
            "Pastikan DATASET_ROOT mengarah ke folder induk yang berisi subfolder kelas."
        )

    rows = []
    for class_dir in sorted([p for p in root_path.iterdir() if p.is_dir()]):
        for f in sorted(class_dir.iterdir()):
            if f.is_file() and f.suffix.lower() in extensions:
                rows.append({
                    "path": str(f),
                    "class": class_dir.name,
                    "filename": f.name,
                    "suffix": f.suffix.lower()
                })

    df = pd.DataFrame(rows, columns=["path", "class", "filename", "suffix"])
    if df.empty:
        print(f"Ã¢Å¡Â Ã¯Â¸Â Tidak ada gambar ditemukan di {root_path}")
    return df


In [ ]:

df_audit = create_initial_audit(DATASET_ROOT)
print(f"Ã°Å¸â€œÅ  Total gambar ditemukan untuk diaudit: {len(df_audit)}")

In [ ]:
def plotting_distribusi_kelas(df_show):



    # 1. Menghitung distribusi frekuensi tiap kelas
    class_counts = df_show['class'].value_counts().reset_index()
    class_counts.columns = ['Nama Kelas', 'Jumlah Sampel']

    # 2. Plotting menggunakan Seaborn
    plt.figure(figsize=(12, 6))
    # Mengurutkan berdasarkan jumlah terbanyak agar lebih mudah dianalisis
    sns.set_style("whitegrid")
    ax = sns.barplot(
        data=class_counts.sort_values('Jumlah Sampel', ascending=False), 
        x='Nama Kelas', 
        y='Jumlah Sampel', 
        palette='viridis',
        hue='Nama Kelas',  # Warna berbeda untuk setiap kelas
        legend=False
    )

    # 3. Menambahkan Label Angka di Atas Setiap Batang (Presisi)
    for p in ax.patches:
        ax.annotate(format(p.get_height(), '.0f'), 
                    (p.get_x() + p.get_width() / 2., p.get_height()), 
                    ha = 'center', va = 'center', 
                    xytext = (0, 9), 
                    textcoords = 'offset points',
                    fontsize=11, fontweight='bold')

    # 4. Estetika Grafik
    plt.title("Distribusi Jumlah Data per Kelas (Initial Audit)", fontsize=15, pad=20)
    plt.xlabel("Kategori / Kelas", fontsize=12)
    plt.ylabel("Total Gambar", fontsize=12)
    plt.xticks(rotation=45) # Memiringkan teks kelas jika namanya panjang
    plt.tight_layout()

    # Simpan untuk laporan (Opsional)
    plt.savefig("distribusi_kelas_awal.png", dpi=300)
    plt.show()

# Referensi Sitasi (IEEE):
# [1] F. G. S. Lopes, et al., "The use of fractal dimension in the exploratory analysis of complex data," Expert Syst. Appl., vol. 164, 2021.

In [ ]:
plotting_distribusi_kelas(df_audit)

In [ ]:
def _compute_phash(path, hash_size=8):
    """Menghasilkan perceptual hash untuk satu gambar."""
    try:
        with Image.open(path) as img:
            img = ImageOps.exif_transpose(img).convert("RGB")
            return str(imagehash.phash(img, hash_size=hash_size))
    except Exception as e:
        return f"ERROR::{e}"

def find_duplicates_by_phash(df, distance_threshold=0, workers=HASH_WORKERS):
    """
    Audit duplikasi berbasis pHash.
    - distance_threshold=0 berarti exact pHash duplicate.
    - threshold > 0 berarti near-duplicate berdasarkan Hamming distance.
    Fungsi ini tidak menghapus file; hanya menandai baris duplikat.
    """
    if df.empty:
        out = df.copy()
        out["phash"] = []
        out["is_duplicate"] = []
        out["duplicate_of"] = []
        return out, []

    work_df = df.copy().reset_index(drop=True)
    paths = work_df["path"].tolist()

    print(f"Ã°Å¸â€Å½ Memulai audit duplikasi pHash | threshold={distance_threshold} | workers={workers}")
    hashes = Parallel(n_jobs=workers)(
        delayed(_compute_phash)(p) for p in tqdm(paths, desc="pHash")
    )

    work_df["phash"] = hashes
    work_df["is_duplicate"] = False
    work_df["duplicate_of"] = ""

    duplicates = []
    seen_exact = {}
    seen_hashes = []  # list of (hash_obj, path)

    for idx, row in work_df.iterrows():
        h_str = row["phash"]
        path = row["path"]

        if str(h_str).startswith("ERROR::"):
            continue

        if distance_threshold == 0:
            if h_str in seen_exact:
                work_df.at[idx, "is_duplicate"] = True
                work_df.at[idx, "duplicate_of"] = seen_exact[h_str]
                duplicates.append({"path": path, "duplicate_of": seen_exact[h_str], "phash": h_str})
            else:
                seen_exact[h_str] = path
        else:
            h_obj = imagehash.hex_to_hash(h_str)
            matched = None
            for prev_hash, prev_path in seen_hashes:
                if h_obj - prev_hash <= distance_threshold:
                    matched = prev_path
                    break

            if matched:
                work_df.at[idx, "is_duplicate"] = True
                work_df.at[idx, "duplicate_of"] = matched
                duplicates.append({"path": path, "duplicate_of": matched, "phash": h_str})
            else:
                seen_hashes.append((h_obj, path))

    return work_df, duplicates

# Eksekusi audit duplikasi tahap 1 pada data original
df_audit, list_duplikat = find_duplicates_by_phash(
    df_audit,
    distance_threshold=PHASH_DISTANCE_THRESHOLD,
    workers=HASH_WORKERS
)


In [ ]:
def plotting_distribusi_duplikat_perkelas(df_duplikat, list_duplikat):

    # Filter hanya data yang duplikat
    dup_stats = df_duplikat[df_duplikat['is_duplicate'] == True]['class'].value_counts().reset_index()
    dup_stats.columns = ['class', 'count']

    if not dup_stats.empty:
        plt.figure(figsize=(12, 6))
        ax = sns.barplot(data=dup_stats, x='class', y='count', palette='Reds_r',    hue='class',  # Warna berbeda untuk setiap kelas
        legend=False)

        # Menambahkan angka di atas batang
        for p in ax.patches:
            ax.annotate(f'{int(p.get_height())}', 
                        (p.get_x() + p.get_width() / 2., p.get_height()), 
                        ha='center', va='center', xytext=(0, 10), 
                        textcoords='offset points', fontsize=11, fontweight='bold')

        plt.title("Jumlah Duplikasi per Kelas (Hasil Audit Tahap Awal)", fontsize=14)
        plt.xlabel("Nama Kelas")
        plt.ylabel("Jumlah Duplikat")
        plt.show()
        print(f"Ã¢Å“â€¦ Total duplikat ditemukan: {len(list_duplikat)}")
    else:
        print("Ã¢Å“â€¦ Luar biasa! Tidak ditemukan duplikasi pada tahap ini.")

In [ ]:
plotting_distribusi_duplikat_perkelas(df_audit, list_duplikat)

In [ ]:
df_duplikasi_cleaned = df_audit[df_audit['is_duplicate'] == False].reset_index(drop=True)


In [ ]:

print(f"Ã¢Å“â€¦ Data duplikat telah berhasil dihapus.")
print(f"Ã°Å¸â€œÅ  Jumlah data bersih sekarang: {len(df_duplikasi_cleaned)}")
print(f"Ã°Å¸â€œÅ  Jumlah data duplikat yang dihapus: {len(df_audit) - len(df_duplikasi_cleaned)}")


In [ ]:
plotting_distribusi_kelas(df_duplikasi_cleaned)

In [ ]:
def process_single_image(input_path: str, output_path: str, target_size=(128, 128)):
    """
    Melakukan Grayscale, Padding, dan Resize pada satu file.
    """
    try:
        with Image.open(input_path).convert('L') as img:
            # 1. Resize dengan menjaga Aspect Ratio (Thumbnail)
            img.thumbnail(target_size, Image.Resampling.LANCZOS)
            
            # 2. Membuat Canvas Hitam untuk Padding (Background)
            canvas = Image.new("L", target_size, 0)
            
            # 3. Menempelkan gambar di tengah (Centering)
            offset = ((target_size[0] - img.size[0]) // 2, 
                      (target_size[1] - img.size[1]) // 2)
            canvas.paste(img, offset)
            
            # 4. Simpan sebagai PNG Lossless
            canvas.save(output_path, "PNG")
            return True
    except Exception as e:
        # Opsional: print(f"Ã¢ÂÅ’ Error pada {input_path}: {e}")
        return False

class PreprocessDataset:
    """
    Dataset sederhana berbasis list task, tanpa dependensi PyTorch.
    """
    def __init__(self, tasks):
        self.tasks = tasks
        
    def __len__(self):
        return len(self.tasks)
        
    def __getitem__(self, idx):
        # Mengembalikan tuple (input_path, output_path)
        return self.tasks[idx]

In [ ]:
def process_single_image(input_path: str, output_path: str, target_size=TARGET_SIZE):
    """Grayscale + padding + resize menjadi PNG lossless."""
    try:
        with Image.open(input_path) as img:
            img = ImageOps.exif_transpose(img).convert("L")
            img.thumbnail(target_size, Image.Resampling.LANCZOS)

            canvas = Image.new("L", target_size, 0)
            offset = ((target_size[0] - img.size[0]) // 2,
                      (target_size[1] - img.size[1]) // 2)
            canvas.paste(img, offset)

            output_path = Path(output_path)
            output_path.parent.mkdir(parents=True, exist_ok=True)
            canvas.save(output_path, "PNG")
            return {"input_path": input_path, "output_path": str(output_path), "status": "ok"}
    except Exception as e:
        return {"input_path": input_path, "output_path": output_path, "status": "error", "error": str(e)}

def run_preprocessing_from_df(df, output_folder=OUTPUT_DIR, size=TARGET_SIZE, workers=PREPROCESS_WORKERS, reset_output_dir=RESET_OUTPUT_DIR):
    output_folder = Path(output_folder)

    if reset_output_dir and output_folder.exists():
        print(f"Ã°Å¸Â§Â¹ RESET_OUTPUT_DIR=True, menghapus folder output lama: {output_folder}")
        shutil.rmtree(output_folder)

    output_folder.mkdir(parents=True, exist_ok=True)

    tasks = []
    print(f"Ã°Å¸â€â€ž Menyiapkan tugas dari {len(df)} baris data bersih tahap 1...")

    for _, row in df.sort_values(["class", "filename"]).iterrows():
        img_p = Path(row["path"])
        class_name = row["class"]
        out_p = output_folder / class_name / (img_p.stem + ".png")
        tasks.append((str(img_p), str(out_p), size))

    print(f"Ã°Å¸Å¡â‚¬ Memulai preprocessing {len(tasks)} gambar | workers={workers}")
    results = Parallel(n_jobs=workers)(
        delayed(process_single_image)(inp, out, sz)
        for inp, out, sz in tqdm(tasks, desc="Preprocessing")
    )

    log_df = pd.DataFrame(results)
    log_df.to_csv("preprocessing_log.csv", index=False)

    ok_count = (log_df["status"] == "ok").sum() if not log_df.empty else 0
    err_count = (log_df["status"] == "error").sum() if not log_df.empty else 0
    print(f"Ã°Å¸ÂÂ Preprocessing selesai. OK={ok_count}, Error={err_count}")
    print(f"Ã°Å¸â€œÂ Output tersimpan di: {output_folder}")

    if err_count:
        display(log_df[log_df["status"] == "error"].head(20))

    return log_df


In [ ]:
# --- EKSEKUSI PREPROCESSING ---
df_duplikasi_cleaned = df_audit[df_audit["is_duplicate"] == False].reset_index(drop=True)

preprocess_log = run_preprocessing_from_df(
    df=df_duplikasi_cleaned,
    output_folder=OUTPUT_DIR,
    size=TARGET_SIZE,
    workers=PREPROCESS_WORKERS,
    reset_output_dir=RESET_OUTPUT_DIR
)


In [ ]:
df_preprocessed = create_initial_audit(OUTPUT_DIR)
print(f"Ã°Å¸â€œÅ  Total gambar hasil preprocessing yang ditemukan: {len(df_preprocessed)}")


In [ ]:

def visualize_df_images(df, n_samples=10, cols=5, figsize=(20, 10)):
    """
    Menampilkan sampel gambar dari DataFrame.
    Argumen:
        df: DataFrame yang berisi kolom 'path' dan 'class'.
        n_samples: Jumlah gambar yang ingin ditampilkan.
        cols: Jumlah kolom dalam grid visualisasi.
        figsize: Ukuran figure matplotlib.
    """
    # 1. Mengambil sampel secara acak dari DataFrame
    # Jika n_samples lebih besar dari jumlah data, ambil semua data
    actual_samples = min(n_samples, len(df))
    sample_df = df.sample(n=actual_samples, random_state=42)
    
    # 2. Menghitung jumlah baris yang dibutuhkan
    rows = (actual_samples + cols - 1) // cols
    
    # 3. Setup Plot
    plt.figure(figsize=figsize)
    
    for i, (idx, row) in enumerate(sample_df.iterrows()):
        plt.subplot(rows, cols, i + 1)
        
        # Load gambar menggunakan path dari DF
        try:
            img = Image.open(row['path'])
            plt.imshow(img, cmap='gray') # Gunakan cmap='gray' jika sudah grayscale
            
            # Menampilkan info kelas dan dimensi pada title
            plt.title(f"Class: {row['class']}\nSize: {img.size}", fontsize=10)
            plt.axis('off')
        except Exception as e:
            plt.title(f"Error loading image")
            plt.axis('off')
            
    plt.tight_layout()
    plt.show()



In [ ]:
# --- CARA EKSEKUSI ---
# Panggil fungsi ini menggunakan df_cleaned Anda
visualize_df_images(df_preprocessed, n_samples=10, cols=5)

In [ ]:
# Audit duplikasi tahap 2 pada citra hasil preprocessing.
# Tahap ini menangkap canonical duplicate setelah grayscale/padding/resize.
df_preprocessed_audit, list_duplikat_stage2 = find_duplicates_by_phash(
    df_preprocessed,
    distance_threshold=PHASH_DISTANCE_THRESHOLD,
    workers=HASH_WORKERS
)


In [ ]:
plotting_distribusi_duplikat_perkelas(df_preprocessed_audit, list_duplikat_stage2)

In [ ]:
df_duplikasi = df_preprocessed_audit[df_preprocessed_audit['is_duplicate'] == True].reset_index(drop=True)


In [ ]:
df_duplikasi.head()

In [ ]:
def execute_direct_cleanup(df, dry_run=False):
    """
    Menghapus file hasil preprocessing yang ditandai sebagai duplikat.
    Hanya digunakan pada OUTPUT_DIR, bukan dataset original.
    """
    if "path" not in df.columns:
        raise ValueError("Kolom 'path' tidak ditemukan dalam DataFrame.")

    deleted_count = 0
    not_found_count = 0
    error_count = 0

    print(f"Ã°Å¸Å¡â‚¬ Memulai cleanup {len(df)} file duplikat stage 2 | dry_run={dry_run}")

    for file_path in tqdm(df["path"], desc="Cleanup duplikat", unit="file"):
        file_path = Path(file_path)
        if dry_run:
            continue

        if file_path.exists():
            try:
                file_path.unlink()
                deleted_count += 1
            except Exception as e:
                print(f"\n[!] Gagal menghapus {file_path}: {e}")
                error_count += 1
        else:
            not_found_count += 1

    summary = {
        "deleted": deleted_count,
        "not_found": not_found_count,
        "error": error_count,
        "dry_run": dry_run
    }
    print("Ã¢Å“â€¦ Cleanup selesai:", summary)
    return summary

# Eksekusi cleanup hanya untuk duplikat di folder OUTPUT_DIR.
df_duplikasi = df_preprocessed_audit[df_preprocessed_audit["is_duplicate"] == True].reset_index(drop=True)
cleanup_summary = execute_direct_cleanup(df_duplikasi, dry_run=False)


In [ ]:
# Audit final setelah cleanup fisik.
df_final = create_initial_audit(OUTPUT_DIR)
df_final = df_final.sort_values(["class", "filename"]).reset_index(drop=True)
df_final.to_csv(FINAL_MANIFEST_CSV, index=False)

print("Ã¢Å“â€¦ Dataset final telah diaudit ulang setelah deduplikasi tahap 2.")
print(f"Ã°Å¸â€œÅ  Jumlah data bersih final: {len(df_final)}")
print(f"Ã°Å¸â€œâ€ž Manifest final disimpan ke: {FINAL_MANIFEST_CSV}")
display(df_final["class"].value_counts().rename_axis("class").reset_index(name="count"))


In [ ]:
plotting_distribusi_kelas(df_final)